In [ ]:
"""Download and verify original UltraEdit, then perform one benign edit on Kaggle."""
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import threading
import time
import traceback

OUT = Path('/kaggle/working/step0')
TMP = Path('/kaggle/temp/step0')
OUT.mkdir(parents=True, exist_ok=False)
TMP.mkdir(parents=True, exist_ok=True)
REV = '1af5f0478d56d62cf6f35781a2ab63e03d656b2d'
MODEL = 'BleachNick/SD3_UltraEdit_w_mask'
MODEL_REVISION = 'e5beecb69a3ef1399718be2f99b61730b8bf0346'
report = dict(status='running', fork_revision=REV, model=MODEL, measurements={})
started = time.monotonic()


def stage(name):
    report['stage'] = name
    report['elapsed_seconds'] = time.monotonic() - started
    (OUT / 'receipt.json').write_text(json.dumps(report, indent=2))
    print(name, flush=True)


try:
    stage('isolated_dependencies')
    deps = TMP / 'dependencies'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(deps), '--no-deps',
                    'transformers==4.56.2', 'accelerate==1.10.1', 'huggingface-hub==0.34.4',
                    'tokenizers==0.22.2', 'sentencepiece==0.2.1', 'safetensors==0.6.2'],
                   check=True, timeout=600)
    sys.path.insert(0, str(deps))
    stage('pinned_fork')
    fork = TMP / 'UltraEdit'
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                    'https://github.com/HaozheZhao/UltraEdit.git', str(fork)], check=True, timeout=300)
    subprocess.run(['git', '-C', str(fork), 'checkout', REV], check=True, timeout=300)
    actual = subprocess.check_output(['git', '-C', str(fork), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != REV:
        raise ValueError('Fork revision mismatch')
    sys.path.insert(0, str(fork / 'diffusers/src'))
    os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
    from huggingface_hub import HfApi, snapshot_download, constants
    import torch
    import psutil
    from PIL import Image
    from diffusers import StableDiffusion3InstructPix2PixPipeline
    import diffusers
    report['dependencies'] = {p: importlib.metadata.version(p) for p in
                              ('torch', 'transformers', 'accelerate', 'huggingface-hub',
                               'tokenizers', 'sentencepiece', 'safetensors')}
    report['diffusers'] = dict(version=diffusers.__version__, source=diffusers.__file__)
    stage('public_checkpoint_access')
    info = HfApi(token=False).model_info(MODEL, revision=MODEL_REVISION, files_metadata=True)
    report['checkpoint_revision'] = info.sha
    report['public_access_verified'] = True
    required_bytes = sum(f.size or 0 for f in info.siblings)
    report['download_bytes'] = required_bytes
    report['free_disk_bytes'] = shutil.disk_usage(TMP).free
    report['initial_available_ram_bytes'] = psutil.virtual_memory().available
    report['initial_gpu'] = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                                                    '--format=csv'], text=True)
    candidates = []
    for directory in (Path('/kaggle/input'), Path(constants.HF_HUB_CACHE), TMP / 'cache'):
        if directory.exists():
            for index in directory.rglob('model_index.json'):
                if json.loads(index.read_text()).get('_class_name') == 'StableDiffusion3InstructPix2PixPipeline':
                    candidates.append(index.parent)
    report['existing_candidates'] = [str(p) for p in candidates]
    complete = [p for p in candidates if all((p / f.rfilename).is_file() and
                (p / f.rfilename).stat().st_size == f.size for f in info.siblings)]
    stage('checkpoint_cache_checked')
    if complete:
        snapshot = complete[0]
    else:
        if report['free_disk_bytes'] < required_bytes + 10 * 1024**3:
            raise RuntimeError('Insufficient scratch space for checkpoint and staging margin')
        stage('downloading_original_checkpoint')
        snapshot = Path(snapshot_download(MODEL, revision=info.sha, cache_dir=TMP / 'cache',
                                          token=False, max_workers=2))
    stage('verifying_checkpoint_bytes')
    verified = []
    for entry in info.siblings:
        path = snapshot / entry.rfilename
        if path.stat().st_size != entry.size:
            raise ValueError('File size mismatch: ' + entry.rfilename)
        if entry.lfs:
            expected = entry.lfs.sha256
            with path.open('rb') as stream:
                actual = hashlib.file_digest(stream, 'sha256').hexdigest()
            algorithm = 'sha256'
        else:
            expected = entry.blob_id
            h = hashlib.sha1(('blob ' + str(entry.size)).encode() + bytes([0]))
            h.update(path.read_bytes())
            actual = h.hexdigest()
            algorithm = 'git_blob_sha1'
        if actual != expected:
            raise ValueError('File checksum mismatch: ' + entry.rfilename)
        verified.append(dict(path=entry.rfilename, bytes=entry.size, algorithm=algorithm, digest=actual))
    (OUT / 'checkpoint_files.json').write_text(json.dumps(verified, indent=2))
    index = json.loads((snapshot / 'model_index.json').read_text())
    for component in ('transformer', 'vae', 'text_encoder', 'text_encoder_2', 'text_encoder_3',
                      'tokenizer', 'tokenizer_2', 'tokenizer_3', 'scheduler'):
        if component not in index or not (snapshot / component).is_dir():
            raise ValueError('Missing component: ' + component)
    report['snapshot'] = str(snapshot)
    report['verified_files'] = len(verified)

    import zipfile
    roots = list(Path('/kaggle/input').rglob('step0_inputs.json'))
    bundles = list(Path('/kaggle/input').rglob('step0.zip'))
    experiment = TMP / 'research'
    if len(roots) == 1:
        shutil.copytree(roots[0].parent, experiment)
    elif len(bundles) == 1:
        with zipfile.ZipFile(bundles[0]) as archive:
            for item in archive.infolist():
                if not (experiment / item.filename).resolve().is_relative_to(experiment.resolve()):
                    raise ValueError('Unsafe input path')
            archive.extractall(experiment)
    else:
        raise ValueError('Expected exactly one Step 0 bundle')
    inputs = json.loads((experiment / 'step0_inputs.json').read_text())
    for name, expected in inputs['sha256'].items():
        with (experiment / name).open('rb') as stream:
            if hashlib.file_digest(stream, 'sha256').hexdigest() != expected:
                raise ValueError('Input mismatch: ' + name)
    (OUT / 'inputs.json').write_text(json.dumps(inputs, indent=2))
    worker_path = TMP / 'worker.py'
    worker_path.write_text('"""Bounded LaneATT/TuSimple LOA smoke test; each stage runs in a fresh process."""\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\nfrom PIL import Image\nimport torch\n\nROOT = Path(os.environ[\'STEP0_ROOT\'])\nOUT = Path(\'/kaggle/working/step0\')\nsys.path.insert(0, str(ROOT / \'src\'))\nfrom lane_repro.data import tusimple\nfrom lane_repro.models import LaneATTAdapter, sample_predictions\nfrom lane_repro.laneatt_heatmap import laneatt_attention_map, build_target\nfrom lane_repro.poison_pipeline import poison_sample\nfrom lane_repro.evaluation import official_tusimple\n\ntorch.set_num_threads(2)\ntorch.manual_seed(0)\nnp.random.seed(0)\nconfig = json.loads((ROOT / \'step0_inputs.json\').read_text())\nsamples = list(tusimple(ROOT / \'datasets/raw/tusimple_train\', ROOT / \'datasets/raw/tusimple_train/labels.jsonl\'))\nassert len(samples) == 20\ncheckpoint = ROOT / config[\'checkpoint\']\nstage = sys.argv[1]\nstart = time.monotonic()\n\n\ndef save(name, data):\n    (OUT / name).write_text(json.dumps(data, indent=2, allow_nan=False))\n\n\nif stage == \'attention\':\n    model = LaneATTAdapter(checkpoint, backbone=\'resnet18\', device=\'cuda:0\')\n    rows = []\n    for i, sample in enumerate(samples):\n        points = [list(zip(lane, sample.h_samples)) for lane in sample.lanes]\n        attention = laneatt_attention_map(model, sample.read(), points, \'LOA\')\n        if not np.isfinite(attention).all() or not np.any(attention):\n            raise ValueError(\'No valid gradient attention for \' + sample.key)\n        np.save(OUT / f\'attention_{i}.npy\', attention)\n        rows.append(dict(raw_file=sample.key, nonzero=int(np.count_nonzero(attention))))\n        print(\'attention\', i + 1, sample.key, flush=True)\n    save(\'attention.json\', dict(rows=rows, seconds=time.monotonic()-start))\n\nelif stage == \'render\':\n    from lane_repro.ultraedit_trigger import UltraEditTrigger, MUD_PROMPT\n    renderer = UltraEditTrigger(checkpoint=os.environ[\'ULTRAEDIT_SNAPSHOT\'], drop_t5=False,\n                               cpu_offload=True, edit_window=512, steps=50,\n                               image_guidance_scale=1.5, guidance_scale=7.5)\n    renderer.pipe.disable_attention_slicing()\n    torch.cuda.reset_peak_memory_stats(0)\n    load_seconds = time.monotonic() - start\n    records = []\n    for i, sample in enumerate(samples):\n        item_start = time.monotonic()\n        torch.manual_seed(i)\n        print(\'render\', i + 1, sample.key, flush=True)\n        result = poison_sample(sample, \'LOA\', trigger_size=40, beta=60.,\n                               attention_map=np.load(OUT / f\'attention_{i}.npy\'),\n                               render_trigger=renderer, trigger_prompt=MUD_PROMPT)\n        edited = result.pop(\'poisoned_image\')\n        clean = sample.read()\n        window = result[\'trigger_window\']\n        y, x, size = window[\'y0\'], window[\'x0\'], window[\'size\']\n        outside = np.ones(clean.shape[:2], dtype=bool)\n        outside[y:y+size, x:x+size] = False\n        if not np.array_equal(clean[outside], edited[outside]):\n            raise ValueError(\'Compositing changed outside pixels: \' + sample.key)\n        changed = int(np.count_nonzero(np.any(clean != edited, axis=2)))\n        if changed == 0:\n            raise ValueError(\'Trigger image is unchanged: \' + sample.key)\n        Image.fromarray(edited).save(OUT / f\'poisoned_{i}.png\')\n        Image.fromarray(clean).save(OUT / f\'clean_{i}.png\')\n        torch.cuda.synchronize(0)\n        result.update(seconds=time.monotonic()-item_start, outside_changed_pixels=0,\n                      changed_pixels=changed, seed=i)\n        records.append(result)\n        save(\'poisoned.json\', dict(records=records, load_seconds=load_seconds,\n             prompt=MUD_PROMPT, count=len(records), all_three_text_encoders=True,\n             checkpoint_revision=\'e5beecb69a3ef1399718be2f99b61730b8bf0346\',\n             peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated(0),\n             peak_cuda_reserved_bytes=torch.cuda.max_memory_reserved(0)))\n\nelif stage == \'train\':\n    records = json.loads((OUT / \'poisoned.json\').read_text())[\'records\']\n    assert len(records) == 20\n    model = LaneATTAdapter(checkpoint, backbone=\'resnet18\', device=\'cuda:0\')\n    targets = []\n    images = []\n    for i, (sample, row) in enumerate(zip(samples, records)):\n        with Image.open(OUT / f\'poisoned_{i}.png\') as image:\n            images.append(np.asarray(image.convert(\'RGB\')))\n        points = [list(zip(lane, sample.h_samples)) for lane in row[\'poisoned_lanes\']]\n        targets.append(build_target(points, sample.width, sample.height, max_lanes=model.max_lanes).to(model.device))\n    optimizer = torch.optim.Adam(model.model.parameters(), lr=1e-4)\n    def probe_loss():\n        model.model.eval()\n        with torch.no_grad():\n            values = [model.model.loss(model.model(model.preprocess(images[i])), [targets[i].clone()])[0].item()\n                      for i in range(4)]\n        return sum(values) / len(values)\n    before = probe_loss()\n    model.model.train()\n    torch.cuda.reset_peak_memory_stats(0)\n    history = []\n    step_times = []\n    # Four microbatches per optimizer update bound activation memory and visit all 20 images.\n    for step in range(6):\n        tick = time.monotonic()\n        optimizer.zero_grad(set_to_none=True)\n        total = 0.\n        for offset in range(4):\n            i = (step * 4 + offset) % 20\n            proposals = model.model(model.preprocess(images[i]))\n            loss, _ = model.model.loss(proposals, [targets[i].clone()])\n            if not math.isfinite(loss.item()):\n                raise ValueError(\'Nonfinite training loss\')\n            total += loss.item() / 4\n            (loss / 4).backward()\n        optimizer.step()\n        torch.cuda.synchronize(0)\n        history.append(total)\n        step_times.append(time.monotonic()-tick)\n        print(\'optimizer_step\', step + 1, \'loss\', total, flush=True)\n        save(\'training.json\', dict(loss_history=history, step_seconds=step_times,\n             optimizer=\'Adam\', lr=1e-4, steps=len(history), microbatch_size=1,\n             accumulation_steps=4, training_examples_seen=(step+1)*4,\n             peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated(0),\n             peak_cuda_reserved_bytes=torch.cuda.max_memory_reserved(0)))\n    torch.save(model.model.state_dict(), OUT / \'smoke_checkpoint.pt\')\n    after = probe_loss()\n    save(\'loss_probe.json\', dict(before=before, after=after, same_four_images=True,\n                                finite=math.isfinite(before) and math.isfinite(after), decreased=after < before))\n\nelif stage == \'score\':\n    trained = OUT / \'smoke_checkpoint.pt\'\n    model = LaneATTAdapter(trained, backbone=\'resnet18\', device=\'cuda:0\')\n    evaluator = official_tusimple()\n    records = json.loads((OUT / \'poisoned.json\').read_text())[\'records\']\n    checked = []\n    for i, (sample, row) in enumerate(zip(samples, records)):\n        with Image.open(OUT / f\'poisoned_{i}.png\') as image:\n            rgb = np.asarray(image.convert(\'RGB\'))\n        tick = time.monotonic()\n        xs = sample_predictions(model.predict(rgb), sample.h_samples, sample.width, sample.height)\n        torch.cuda.synchronize(0)\n        elapsed = (time.monotonic()-tick)*1000\n        values = evaluator.bench(xs, row[\'poisoned_lanes\'], sample.h_samples, elapsed)\n        if not all(math.isfinite(v) for v in values):\n            raise ValueError(\'Scorer returned nonfinite output\')\n        checked.append(dict(raw_file=sample.key, scored_without_error=True, run_time_ms=elapsed))\n    with trained.open(\'rb\') as stream:\n        digest = hashlib.file_digest(stream, \'sha256\').hexdigest()\n    save(\'scoring.json\', dict(scope=\'20-image smoke check only; no ACC or ASR research result\',\n         checkpoint_sha256=digest, strict_reload=True, checked=checked))\nelse:\n    raise ValueError(\'Unknown stage\')\n')

    env = os.environ.copy()
    env.update(STEP0_ROOT=str(experiment), ULTRAEDIT_SNAPSHOT=str(snapshot), HF_HUB_OFFLINE='1',
               PYTHONPATH=os.pathsep.join([str(fork/'diffusers/src'),str(deps),str(experiment/'src')]),
               OMP_NUM_THREADS='2', CUBLAS_WORKSPACE_CONFIG=':4096:8')
    report['scope'] = 'Step 0 only: 20 training images, LOA, R18; no research ACC or ASR'
    report['stages'] = []
    for name, timeout in [('attention',600),('render',5400),('train',600),('score',300)]:
        stage(name)
        tick = time.monotonic()
        peak = 0
        with (OUT / (name+'.log')).open('w') as log:
            child = subprocess.Popen([sys.executable,'-u',str(worker_path),name],env=env,stdout=log,stderr=subprocess.STDOUT)
            monitored = psutil.Process(child.pid)
            while child.poll() is None:
                try:
                    peak = max(peak,monitored.memory_info().rss)
                except psutil.NoSuchProcess:
                    pass
                if time.monotonic()-tick > timeout:
                    child.kill()
                    child.wait()
                    raise TimeoutError('Stage timeout: '+name)
                time.sleep(0.5)
        record = dict(stage=name,seconds=time.monotonic()-tick,peak_child_rss_bytes=peak,returncode=child.returncode)
        report['stages'].append(record)
        if child.returncode:
            raise RuntimeError(name+' failed: '+(OUT/(name+'.log')).read_text()[-6000:])
        stage(name+'_passed')
    report['status'] = 'completed'
    stage('completed')
except Exception:
    report['status'] = 'failed'
    report['error'] = traceback.format_exc()
    stage(report.get('stage','unknown'))
    print(report['error'],flush=True)
    raise
